In [ ]:
#!/usr/bin/env python3
"""Deploy and run one four-replica Shabdiz experiment on GCP.

The script creates or reuses four replica VMs plus one client VM, prepares the
stellar-private configuration, starts Shabdiz, runs shab_client, and downloads
all replica and client logs into one timestamped result directory.

Run this script from the root of the modified stellar-core repository.
"""

from __future__ import annotations

import concurrent.futures
import posixpath
import shutil
import subprocess
import time
from datetime import datetime
from pathlib import Path
from typing import Callable, Iterable, Sequence, TypeVar


T = TypeVar("T")
R = TypeVar("R")


# =============================================================================
# User configuration
# =============================================================================

PROJECT = "research-488322"
ZONE = "us-west1-b"
MACHINE_TYPE = "e2-standard-2"
IMAGE_FAMILY = "tsm-sc-family"
SUBNET = "default"
SERVICE_ACCOUNT = "254510644191-compute@developer.gserviceaccount.com"

REMOTE_USER = "tejas"
REMOTE_HOME = f"/home/{REMOTE_USER}"
REMOTE_REPO = f"{REMOTE_HOME}/stellar-core"
REMOTE_PRIVATE_DIR = f"{REMOTE_HOME}/stellar-private"

INSTANCE_PREFIX = "tsm-sc-"
NUM_REPLICAS = 4
NUM_CLIENT_VMS = 1

# Client load: aggregate maximum in-flight = 1 * 2 * 150 = 300.
ACTIVE_CLIENTS = 1
CLIENT_THREADS = 2
CLIENT_MAX_IN_FLIGHT_PER_THREAD = 150
CLIENT_TOTAL_REQUESTS = 100000000
CLIENT_DURATION_SEC = 180
CLIENT_WAIT_AFTER_START_SEC = CLIENT_DURATION_SEC + 30
SERVER_BATCH_SIZE_HINT = 100
SEND_INTERVAL_US = 0
CLIENT_PORT = 12000

REPLICA_STARTUP_WAIT_SEC = 60
VM_STARTUP_WAIT_SEC = 30

# The local repository is assumed to be the current working directory.
LOCAL_REPO = Path.cwd().resolve()
SETUP_SCRIPT = LOCAL_REPO / "gcp_setup_stellar_private.sh"
IP_FILE = LOCAL_REPO / "tsm_ips.txt"
LOCAL_PRIVATE_DIR = (LOCAL_REPO.parent / "stellar-private").resolve()
RESULTS_BASE_DIR = Path.home() / "work" / "experiments" / "shabdiz"
LATEST_RUN_FILE = RESULTS_BASE_DIR / "latest_4node_run.txt"

# Set these based on how you want to manage the VMs.
DELETE_EXISTING_BEFORE_RUN = True
DELETE_VMS_AFTER_RUN = True
PULL_REMOTE_CODE = True
COMPILE_REMOTE_CODE = True

# Copy all four replica logs. For faster collection, use [0, 1, 2].
REPLICA_LOG_INDICES = list(range(NUM_REPLICAS))


# =============================================================================
# Generic helpers
# =============================================================================


def run_command(args: Sequence[str], *, cwd: Path | None = None) -> None:
    """Run a local command and fail immediately if it is unsuccessful."""
    printable = " ".join(str(arg) for arg in args)
    print(f"\n$ {printable}")
    subprocess.run(
        [str(arg) for arg in args],
        cwd=str(cwd) if cwd is not None else None,
        check=True,
    )


def capture_command(args: Sequence[str]) -> str:
    """Run a command and return stripped standard output."""
    printable = " ".join(str(arg) for arg in args)
    print(f"\n$ {printable}")
    return subprocess.check_output([str(arg) for arg in args], text=True).strip()


def run_parallel(
    function: Callable[[T], R],
    items: Iterable[T],
    *,
    max_workers: int,
) -> list[R]:
    item_list = list(items)
    if not item_list:
        return []

    with concurrent.futures.ThreadPoolExecutor(
        max_workers=min(max_workers, len(item_list))
    ) as executor:
        return list(executor.map(function, item_list))


def instance_name(index: int) -> str:
    return f"{INSTANCE_PREFIX}{index:03d}"


def replica_indices() -> range:
    return range(NUM_REPLICAS)


def client_indices() -> range:
    return range(NUM_REPLICAS, NUM_REPLICAS + NUM_CLIENT_VMS)


def all_instance_indices() -> range:
    return range(NUM_REPLICAS + NUM_CLIENT_VMS)


# =============================================================================
# GCP instance management
# =============================================================================


def fetch_existing_instances() -> list[tuple[str, str]]:
    output = capture_command(
        [
            "gcloud",
            "compute",
            "instances",
            "list",
            f"--project={PROJECT}",
            f"--filter=name~^{INSTANCE_PREFIX}",
            "--format=value(name,zone)",
        ]
    )

    instances: list[tuple[str, str]] = []
    for line in output.splitlines():
        if not line.strip():
            continue
        name, zone = line.split()
        instances.append((name, zone))
    return instances


def delete_instance(record: tuple[str, str]) -> None:
    name, zone = record
    run_command(
        [
            "gcloud",
            "compute",
            "instances",
            "delete",
            name,
            f"--zone={zone}",
            f"--project={PROJECT}",
            "--quiet",
        ]
    )


def delete_all_experiment_instances() -> None:
    instances = fetch_existing_instances()
    if not instances:
        print("No existing tsm-sc-* instances were found.")
        return

    print("Deleting existing experiment instances:")
    for name, zone in instances:
        print(f"  {name} ({zone})")

    run_parallel(delete_instance, instances, max_workers=32)


def create_instance(index: int) -> None:
    run_command(
        [
            "gcloud",
            "compute",
            "instances",
            "create",
            instance_name(index),
            f"--project={PROJECT}",
            f"--zone={ZONE}",
            f"--machine-type={MACHINE_TYPE}",
            (
                "--network-interface="
                "network-tier=PREMIUM,stack-type=IPV4_ONLY,"
                f"subnet={SUBNET}"
            ),
            "--can-ip-forward",
            "--maintenance-policy=MIGRATE",
            "--provisioning-model=STANDARD",
            f"--service-account={SERVICE_ACCOUNT}",
            (
                "--scopes="
                "https://www.googleapis.com/auth/devstorage.read_only,"
                "https://www.googleapis.com/auth/logging.write,"
                "https://www.googleapis.com/auth/monitoring.write,"
                "https://www.googleapis.com/auth/service.management.readonly,"
                "https://www.googleapis.com/auth/servicecontrol,"
                "https://www.googleapis.com/auth/trace.append"
            ),
            "--tags=http-server,https-server",
            (
                "--create-disk="
                "auto-delete=yes,boot=yes,"
                f"image-family={IMAGE_FAMILY},mode=rw,size=20,type=pd-balanced"
            ),
            "--no-shielded-secure-boot",
            "--shielded-vtpm",
            "--shielded-integrity-monitoring",
            "--labels=goog-ec-src=vm_add-gcloud",
            "--reservation-affinity=any",
        ]
    )


def create_missing_instances() -> None:
    existing_names = {name for name, _ in fetch_existing_instances()}
    missing_indices = [
        index
        for index in all_instance_indices()
        if instance_name(index) not in existing_names
    ]

    if not missing_indices:
        print("All four replicas and the client VM already exist.")
        return

    print(f"Creating missing VM indices: {missing_indices}")
    run_parallel(create_instance, missing_indices, max_workers=5)
    print(f"Waiting {VM_STARTUP_WAIT_SEC} seconds for SSH readiness...")
    time.sleep(VM_STARTUP_WAIT_SEC)


def get_instance_records() -> list[tuple[int, str, str, str]]:
    output = capture_command(
        [
            "gcloud",
            "compute",
            "instances",
            "list",
            f"--project={PROJECT}",
            f"--filter=name~^{INSTANCE_PREFIX}",
            "--sort-by=name",
            "--format=value(name,zone,networkInterfaces[0].networkIP)",
        ]
    )

    records: list[tuple[int, str, str, str]] = []
    for line in output.splitlines():
        if not line.strip():
            continue
        name, zone, private_ip = line.split()
        index = int(name.rsplit("-", 1)[1])
        if index < NUM_REPLICAS + NUM_CLIENT_VMS:
            records.append((index, name, zone, private_ip))

    records.sort()
    return records


# =============================================================================
# Remote commands and file transfer
# =============================================================================


def ssh(index: int, remote_command: str) -> None:
    run_command(
        [
            "gcloud",
            "compute",
            "ssh",
            instance_name(index),
            f"--zone={ZONE}",
            f"--project={PROJECT}",
            "--command",
            remote_command,
        ]
    )


def kill_processes(index: int) -> None:
    ssh(
        index,
        "sudo pkill -9 stellar-core || true; "
        "sudo pkill -9 shab_client || true",
    )


def pull_code(index: int) -> None:
    ssh(index, f"cd {REMOTE_REPO} && git pull --ff-only")


def compile_code(index: int) -> None:
    ssh(
        index,
        (
            f"cd {REMOTE_REPO} && "
            "g++ -O2 -std=c++17 -pthread "
            f"-I{REMOTE_REPO}/src {REMOTE_REPO}/shab_client.cpp "
            f"-o {REMOTE_REPO}/shab_client && "
            "make -j4 && "
            f"sudo rm -rf {REMOTE_PRIVATE_DIR}"
        ),
    )


def clean_remote_private_dir(index: int) -> None:
    ssh(index, f"sudo rm -rf {REMOTE_PRIVATE_DIR}")


def copy_private_config_to_instance(index: int) -> None:
    run_command(
        [
            "gcloud",
            "compute",
            "scp",
            f"--zone={ZONE}",
            f"--project={PROJECT}",
            "--recurse",
            str(LOCAL_PRIVATE_DIR),
            f"{instance_name(index)}:{REMOTE_PRIVATE_DIR}",
        ]
    )


def start_replica(index: int) -> None:
    node_number = index + 1
    ssh(
        index,
        (
            f"cd {REMOTE_PRIVATE_DIR} && "
            f"nohup {REMOTE_REPO}/src/stellar-core run "
            f"--conf node{node_number}/stellar-core.cfg "
            f"> node{node_number}/stellar-core.log 2>&1 "
            "< /dev/null &"
        ),
    )


def start_client(index: int, leader_ip: str) -> None:
    client_id = index - NUM_REPLICAS
    ssh(
        index,
        (
            f"cd {REMOTE_PRIVATE_DIR} && "
            f"nohup {REMOTE_REPO}/shab_client {leader_ip} {CLIENT_PORT} "
            f"{CLIENT_MAX_IN_FLIGHT_PER_THREAD} {CLIENT_TOTAL_REQUESTS} "
            f"{SERVER_BATCH_SIZE_HINT} {CLIENT_DURATION_SEC} "
            f"{SEND_INTERVAL_US} {CLIENT_THREADS} "
            f"> stellar-client-{client_id}.log 2>&1 "
            "< /dev/null &"
        ),
    )


def copy_replica_log(index: int, run_dir: Path) -> None:
    node_number = index + 1
    destination = run_dir / instance_name(index)
    destination.mkdir(parents=True, exist_ok=True)

    remote_node_dir = posixpath.join(REMOTE_PRIVATE_DIR, f"node{node_number}")
    run_command(
        [
            "gcloud",
            "compute",
            "scp",
            f"--zone={ZONE}",
            f"--project={PROJECT}",
            "--recurse",
            f"{instance_name(index)}:{remote_node_dir}",
            str(destination),
        ]
    )


def copy_client_log(index: int, run_dir: Path) -> None:
    client_id = index - NUM_REPLICAS
    run_command(
        [
            "gcloud",
            "compute",
            "scp",
            f"--zone={ZONE}",
            f"--project={PROJECT}",
            (
                f"{instance_name(index)}:{REMOTE_PRIVATE_DIR}/"
                f"stellar-client-{client_id}.log"
            ),
            str(run_dir / f"client_{client_id}.log"),
        ]
    )


# =============================================================================
# Local Shabdiz configuration and result metadata
# =============================================================================


def prepare_local_private_config(replica_ips: list[str]) -> None:
    if not SETUP_SCRIPT.exists():
        raise FileNotFoundError(
            f"Missing {SETUP_SCRIPT}. Run this script from the stellar-core repo root."
        )

    IP_FILE.write_text("".join(f"{ip}\n" for ip in replica_ips))

    if LOCAL_PRIVATE_DIR.exists():
        shutil.rmtree(LOCAL_PRIVATE_DIR)
    LOCAL_PRIVATE_DIR.mkdir(parents=True)

    copied_setup_script = LOCAL_PRIVATE_DIR / SETUP_SCRIPT.name
    shutil.copy2(SETUP_SCRIPT, copied_setup_script)
    copied_setup_script.chmod(0o755)

    run_command([f"./{SETUP_SCRIPT.name}", "start"], cwd=LOCAL_PRIVATE_DIR)
    run_command([f"./{SETUP_SCRIPT.name}"], cwd=LOCAL_PRIVATE_DIR)

    leader_config = LOCAL_PRIVATE_DIR / "node1" / "stellar-core.cfg"
    if not leader_config.exists():
        raise FileNotFoundError(f"Configuration generator did not create {leader_config}")

    config_text = leader_config.read_text()
    if "SEND_CUSTOM_MESSAGE=true" not in config_text.splitlines():
        leader_config.write_text("SEND_CUSTOM_MESSAGE=true\n" + config_text)


def create_run_directory() -> Path:
    total_threads = ACTIVE_CLIENTS * CLIENT_THREADS
    total_in_flight = total_threads * CLIENT_MAX_IN_FLIGHT_PER_THREAD
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

    run_dir = RESULTS_BASE_DIR / (
        f"shabdiz_4nodes_{timestamp}_"
        f"clients_{ACTIVE_CLIENTS}_threads_{CLIENT_THREADS}_"
        f"inflight_{CLIENT_MAX_IN_FLIGHT_PER_THREAD}_"
        f"total_threads_{total_threads}_total_inflight_{total_in_flight}"
    )
    run_dir.mkdir(parents=True, exist_ok=False)
    return run_dir


def write_run_config(run_dir: Path, leader_ip: str) -> None:
    total_threads = ACTIVE_CLIENTS * CLIENT_THREADS
    total_in_flight = total_threads * CLIENT_MAX_IN_FLIGHT_PER_THREAD

    values = {
        "protocol": "SHABDIZ",
        "num_nodes": NUM_REPLICAS,
        "project": PROJECT,
        "zone": ZONE,
        "machine_type": MACHINE_TYPE,
        "active_clients": ACTIVE_CLIENTS,
        "client_threads_per_vm": CLIENT_THREADS,
        "total_client_threads": total_threads,
        "client_max_in_flight_per_thread": CLIENT_MAX_IN_FLIGHT_PER_THREAD,
        "aggregate_max_in_flight": total_in_flight,
        "client_total_requests": CLIENT_TOTAL_REQUESTS,
        "client_duration_sec": CLIENT_DURATION_SEC,
        "client_wait_after_start_sec": CLIENT_WAIT_AFTER_START_SEC,
        "send_interval_us": SEND_INTERVAL_US,
        "server_batch_size_hint": SERVER_BATCH_SIZE_HINT,
        "leader_ip": leader_ip,
        "client_port": CLIENT_PORT,
    }

    with (run_dir / "run_config.txt").open("w") as config_file:
        for key, value in values.items():
            config_file.write(f"{key}={value}\n")


# =============================================================================
# Main experiment
# =============================================================================


def main() -> None:
    if ACTIVE_CLIENTS > NUM_CLIENT_VMS:
        raise ValueError("ACTIVE_CLIENTS cannot exceed NUM_CLIENT_VMS")

    if DELETE_EXISTING_BEFORE_RUN:
        delete_all_experiment_instances()

    create_missing_instances()

    records = get_instance_records()
    replicas = [record for record in records if record[0] < NUM_REPLICAS]
    clients = [record for record in records if record[0] in client_indices()]

    if len(replicas) != NUM_REPLICAS:
        raise RuntimeError(f"Expected four replicas, found: {replicas}")
    if len(clients) != NUM_CLIENT_VMS:
        raise RuntimeError(f"Expected one client VM, found: {clients}")

    replica_ips = [record[3] for record in replicas]
    leader_ip = replica_ips[0]

    print(f"Replica private IPs: {replica_ips}")
    print(f"Client VM: {clients[0][1]}")
    print(f"Client target: {leader_ip}:{CLIENT_PORT}")

    prepare_local_private_config(replica_ips)

    subprocess.call('git add .; git commit -m "testing"; git push', shell=True)

    # Stop stale processes before changing code or configuration.
    run_parallel(kill_processes, all_instance_indices(), max_workers=5)

    if PULL_REMOTE_CODE:
        run_parallel(pull_code, all_instance_indices(), max_workers=5)

    if COMPILE_REMOTE_CODE:
        run_parallel(compile_code, all_instance_indices(), max_workers=5)

    run_parallel(clean_remote_private_dir, all_instance_indices(), max_workers=5)
    run_parallel(copy_private_config_to_instance, all_instance_indices(), max_workers=5)

    run_dir = create_run_directory()
    write_run_config(run_dir, leader_ip)

    active_client_indices = [NUM_REPLICAS + i for i in range(ACTIVE_CLIENTS)]

    try:
        run_parallel(start_replica, replica_indices(), max_workers=NUM_REPLICAS)
        print(f"Waiting {REPLICA_STARTUP_WAIT_SEC} seconds for replicas...")
        time.sleep(REPLICA_STARTUP_WAIT_SEC)

        run_parallel(
            lambda index: start_client(index, leader_ip),
            active_client_indices,
            max_workers=ACTIVE_CLIENTS,
        )

        print(
            f"Shabdiz is running. Waiting {CLIENT_WAIT_AFTER_START_SEC} seconds "
            "before collecting logs."
        )
        time.sleep(CLIENT_WAIT_AFTER_START_SEC)
    finally:
        # Always stop processes, including after Ctrl+C or a failed subcommand.
        run_parallel(kill_processes, all_instance_indices(), max_workers=5)

    run_parallel(
        lambda index: copy_replica_log(index, run_dir),
        REPLICA_LOG_INDICES,
        max_workers=len(REPLICA_LOG_INDICES),
    )
    run_parallel(
        lambda index: copy_client_log(index, run_dir),
        active_client_indices,
        max_workers=ACTIVE_CLIENTS,
    )

    RESULTS_BASE_DIR.mkdir(parents=True, exist_ok=True)
    LATEST_RUN_FILE.write_text(str(run_dir.resolve()) + "\n")

    print("\nExperiment complete.")
    print(f"Results: {run_dir}")
    print(f"Latest-run marker: {LATEST_RUN_FILE}")
    print(
        "Plot with:\n"
        f"  python3 plot_shabdiz_timeseries.py {run_dir}"
    )

    if DELETE_VMS_AFTER_RUN:
        delete_all_experiment_instances()
    else:
        print("VMs were left running. Set DELETE_VMS_AFTER_RUN=True to remove them.")


if __name__ == "__main__":
    main()


$ gcloud compute instances list --project=research-488322 --filter=name~^tsm-sc- --format=value(name,zone)
Deleting existing experiment instances:
  tsm-sc-000 (us-west1-b)
  tsm-sc-001 (us-west1-b)
  tsm-sc-002 (us-west1-b)
  tsm-sc-003 (us-west1-b)
  tsm-sc-004 (us-west1-b)

$ gcloud compute instances delete tsm-sc-000 --zone=us-west1-b --project=research-488322 --quiet

$ gcloud compute instances delete tsm-sc-001 --zone=us-west1-b --project=research-488322 --quiet

$ gcloud compute instances delete tsm-sc-002 --zone=us-west1-b --project=research-488322 --quiet

$ gcloud compute instances delete tsm-sc-003 --zone=us-west1-b --project=research-488322 --quiet

$ gcloud compute instances delete tsm-sc-004 --zone=us-west1-b --project=research-488322 --quiet


Deleted [https://www.googleapis.com/compute/v1/projects/research-488322/zones/us-west1-b/instances/tsm-sc-004].
Deleted [https://www.googleapis.com/compute/v1/projects/research-488322/zones/us-west1-b/instances/tsm-sc-001].
Deleted [https://www.googleapis.com/compute/v1/projects/research-488322/zones/us-west1-b/instances/tsm-sc-002].
